Importing Libraries

In [2]:
!pip install pypdf

In [3]:
# We are importing the PyPDF library. It helps us read PDF files page by page
import pypdf as pdf

In [5]:
reader = pdf.PdfReader("/content/Employee_Handbook_Sample.pdf")

Document Ingestion

In [6]:
# Returns a list-like collection(VirtualList) of page object.
reader.pages

In [7]:
# Total number of pages in the PDF
len(reader.pages)

1

In [8]:
reader.pages[0]

{'/Contents': {'/Filter': ['/ASCII85Decode', '/FlateDecode']},
 '/MediaBox': [0, 0, 595.2756, 841.8898],
 '/Parent': {'/Count': 1,
  '/Kids': [IndirectObject(4, 0, 137973046702032)],
  '/Type': '/Pages'},
 '/Resources': {'/Font': {'/F1': {'/BaseFont': '/Helvetica',
    '/Encoding': '/WinAnsiEncoding',
    '/Name': '/F1',
    '/Subtype': '/Type1',
    '/Type': '/Font'},
   '/F2': {'/BaseFont': '/Helvetica-Bold',
    '/Encoding': '/WinAnsiEncoding',
    '/Name': '/F2',
    '/Subtype': '/Type1',
    '/Type': '/Font'}},
  '/ProcSet': ['/PDF', '/Text', '/ImageB', '/ImageC', '/ImageI']},
 '/Rotate': 0,
 '/Trans': {},
 '/Type': '/Page'}

In [9]:
page = reader.pages[0] # Open the pdf
text = page.extract_text() # Get One Page
print(text) # Ask the page to give its text

Employee Handbook
Welcome to ABC Technologies. This handbook contains company policies.
Leave Policy
Employees are entitled to 18 casual leaves, 12 sick leaves, and 15 earned leaves per year. Casual
leave requires manager approval.
Working Hours
Office timings are 9:00 AM to 6:00 PM, Monday through Friday.
Work From Home
Employees may work from home up to two days per week with manager approval.
Medical Insurance
The company provides health insurance coverage for employees and eligible dependents.
Travel Policy
Domestic travel requires prior approval. Reimbursements are processed within 10 business days.
Code of Conduct
Employees are expected to maintain professionalism, respect colleagues, and protect confidential
company information.
IT Security
Do not share passwords. Enable multi-factor authentication and report suspicious emails
immediately.
Performance Review
Performance reviews are conducted twice a year based on goals, teamwork, and technical
contributions.
Contact Information


Chunking

In [10]:
!pip install langchain

In [11]:
!pip show langchain

Name: langchain
Version: 1.4.7
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [12]:
!pip install -U langchain-text-splitters

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

In [15]:
chunks = text_splitter.split_text(text)

In [16]:
len(chunks)

2

In [17]:
print(chunks[0]) # First chunk

Employee Handbook
Welcome to ABC Technologies. This handbook contains company policies.
Leave Policy
Employees are entitled to 18 casual leaves, 12 sick leaves, and 15 earned leaves per year. Casual
leave requires manager approval.
Working Hours
Office timings are 9:00 AM to 6:00 PM, Monday through Friday.
Work From Home
Employees may work from home up to two days per week with manager approval.
Medical Insurance
The company provides health insurance coverage for employees and eligible dependents.
Travel Policy
Domestic travel requires prior approval. Reimbursements are processed within 10 business days.
Code of Conduct
Employees are expected to maintain professionalism, respect colleagues, and protect confidential
company information.
IT Security
Do not share passwords. Enable multi-factor authentication and report suspicious emails
immediately.
Performance Review
Performance reviews are conducted twice a year based on goals, teamwork, and technical
contributions.
Contact Information


In [18]:
print("-" * 50)

--------------------------------------------------


In [19]:
print(chunks[1]) # Second Chunk

immediately.
Performance Review
Performance reviews are conducted twice a year based on goals, teamwork, and technical
contributions.
Contact Information
For HR-related queries, email hr@abctech.example.


Embeddings

In [20]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.6 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [22]:
# Import the userdata module
from google.colab import userdata

In [23]:
# This fetches the api from collab secrets
GOOGLE_API_KEY=userdata.get("GEMINI_API_KEY")

In [24]:
# create the embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-2",
    google_api_key = GOOGLE_API_KEY
)

In [25]:
embedding_vector = embeddings.embed_query(chunks[0])

print(type(embedding_vector))
print(len(embedding_vector))
print(embedding_vector[:10])

<class 'list'>
3072
[-0.00071063277, 0.0053558494, -0.0028378183, 0.0028784538, 0.012808794, -0.0047902158, -0.027287405, 0.033818424, 0.0046529686, -0.05585856]


ChromaDB

In [28]:
!pip install -q langchain-chroma chromadb

In [29]:
from langchain_chroma import Chroma
print("Chroma imported successfully")

Chroma imported successfully


In [31]:
# Creating a vector Database
vector_db = Chroma.from_texts(
    texts = chunks,
    embedding = embeddings
)

In [32]:
print(vector_db)

Phase 1 Completed

Phase 2

In [33]:
# Step 1 : CReating the retrieval
retriever = vector_db.as_retriever(
    search_kwargs = {"k" : 3}
)

In [34]:
# Step 2 : Asking a question
query = "How many casual leaves are given?"

In [35]:
# Step 3 : Retrieve Similar Chunks
results = retriever.invoke(query)

In [36]:
print(results)

[Document(id='11161e1a-3407-4c60-83ad-78623a05e8fd', metadata={}, page_content='Employee Handbook\nWelcome to ABC Technologies. This handbook contains company policies.\nLeave Policy\nEmployees are entitled to 18 casual leaves, 12 sick leaves, and 15 earned leaves per year. Casual\nleave requires manager approval.\nWorking Hours\nOffice timings are 9:00 AM to 6:00 PM, Monday through Friday.\nWork From Home\nEmployees may work from home up to two days per week with manager approval.\nMedical Insurance\nThe company provides health insurance coverage for employees and eligible dependents.\nTravel Policy\nDomestic travel requires prior approval. Reimbursements are processed within 10 business days.\nCode of Conduct\nEmployees are expected to maintain professionalism, respect colleagues, and protect confidential\ncompany information.\nIT Security\nDo not share passwords. Enable multi-factor authentication and report suspicious emails\nimmediately.\nPerformance Review\nPerformance reviews ar

Phase 2 Completed

Phase 3

In [37]:
# Setp 1 : install Gemini Integration
from langchain_google_genai import ChatGoogleGenerativeAI

In [41]:
# Step 2 : Create the LLM
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0,
    google_api_key = GOOGLE_API_KEY
)

In [42]:
# Step 3 : Function to convert Documents into plain text
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [43]:
# Step 4 : Ask a question
query = "How many casual leaves are given?"

In [44]:
# Step 5 : Retrieve relevant chunks
results = retriever.invoke(query)

In [45]:
# Step 6 : Convert retrieved documents into text
context = format_docs(results)

In [47]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question ONLY using the context below.

Context:
{context}

Question:
{question}
""")

In [48]:
# Step 7 : Fill the prompt
final_prompt = prompt.invoke(
    {
        "context" : context,
        "question": query
    }
)

In [49]:
# Step 8 : Send it to Gemini
response = llm.invoke(final_prompt)

In [50]:
# Print the answer
print(response.content)

Employees are entitled to 18 casual leaves per year.
